# GS05 — Perception with ROCm in Genesis

### Lab Description

This lab extends the Genesis sequence from scripted motion to perception-guided decisions. Genesis camera buffers are processed with editable ROCm/PyTorch kernels and converted into a structured target record for the language-guided agent in GS06.

The processing pipeline is:

`RGB / depth / segmentation / normal → ROCm kernels → visual target selection → target record`

> **Important:** vision selects the target identity. The exact 3D world position is read from Genesis entity state and labeled as simulator ground truth; it is not presented as monocular 3D reconstruction.

#### Recommended Hardware

An AMD GPU supported by ROCm, such as an AMD Radeon™ GPU or AMD Ryzen™ AI processor with integrated Radeon graphics.

#### Software Environment

OS: Ubuntu 24.04 LTS  
Install [AUP Learning Cloud](https://amdresearch.github.io/aup-learning-cloud/installation/quick-start.html). The Genesis Simulation image provides ROCm, PyTorch, and `genesis-world==1.3.1`.

## Goals

- Capture RGB, depth, segmentation, and normal buffers from Genesis.
- Verify that PyTorch kernels execute on the ROCm/HIP device.
- Implement depth, normal, segmentation, color-lock, grayscale, blur, and Sobel processing.
- Reduce two 8×8 fingertip tactile fields into contact, force, and secure-grasp signals.
- Build an eight-panel perception dashboard.
- Produce visual and tactile contracts for GS06.

In [ ]:
import os
import json
import logging
import warnings
import collections

os.environ.setdefault("TI_LOG_LEVEL", "error")
warnings.filterwarnings("ignore")

import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np
import torch
import torch.nn.functional as F
import genesis as gs
import genesis.utils.geom as gu
from genesis.utils.misc import tensor_to_array

from helpers.physisim_hud import FFmpegHUDWriter, GPUMonitor, compose_hud_frame
from helpers.physisim_widget import LiveHUDController

os.makedirs("Videos", exist_ok=True)
os.makedirs("Artifacts", exist_ok=True)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("=== ROCm runtime receipt ===")
print("PyTorch :", torch.__version__)
print("HIP     :", getattr(torch.version, "hip", None))
print("Device  :", DEVICE)
if DEVICE.type == "cuda":
    print("GPU     :", torch.cuda.get_device_name(0))
else:
    print("WARNING : ROCm/HIP is unavailable; kernels will run on CPU.")

## 1. Build the perception scene

The scene follows the same Genesis 1.3.1 pattern as GS03: a headless scene, Franka Panda, fixed camera, and colored cubes.

> Genesis initialization and scene build should run only once per kernel. Restart the kernel before rerunning this section.

In [ ]:
assert "scene" not in globals(), "Scene already exists. Restart the kernel before rebuilding it."

gs.init(backend=gs.amdgpu, theme="light", seed=0)
gs.logger._logger.setLevel(logging.WARNING)

CUBE_SIZE = 0.04
CUBE_Z = CUBE_SIZE / 2.0

scene = gs.Scene(
    viewer_options=gs.options.ViewerOptions(
        camera_pos=(3, -1, 1.5),
        camera_lookat=(0.0, 0.0, 0.5),
        camera_fov=30,
        max_FPS=60,
    ),
    sim_options=gs.options.SimOptions(dt=0.01, substeps=4),
    rigid_options=gs.options.RigidOptions(
        box_box_detection=True,
        constraint_timeconst=0.01,
    ),
    show_viewer=False,
)

plane = scene.add_entity(gs.morphs.Plane())
franka = scene.add_entity(
    gs.morphs.MJCF(file="xml/franka_emika_panda/panda.xml"),
)

cube_specs = {
    "red":   {"pos": (0.55, -0.15, CUBE_Z), "color": (1.0, 0.0, 0.0, 1.0)},
    "green": {"pos": (0.55,  0.00, CUBE_Z), "color": (0.0, 1.0, 0.0, 1.0)},
    "blue":  {"pos": (0.55,  0.15, CUBE_Z), "color": (0.0, 0.0, 1.0, 1.0)},
}

cube_entities = {}
for name, spec in cube_specs.items():
    cube_entities[name] = scene.add_entity(
        gs.morphs.Box(size=(CUBE_SIZE,) * 3, pos=spec["pos"]),
        surface=gs.surfaces.Default(color=spec["color"]),
    )

cam = scene.add_camera(
    res=(640, 480),
    pos=(3, -1, 1.5),
    lookat=(0, 0, 0.5),
    fov=30,
    GUI=True,
)

# Official Genesis 1.3.1 Franka tactile layout: one 8×8 pad per fingertip.
CONTACT_THRESH_M = 5e-4
CONTACT_SECURE_TAXELS = 12
GRIP_STIFFNESS_N_PER_M = 5000.0

probe_normal = (0.0, -1.0, 0.0)
probe_local_pos = gu.generate_grid_points_on_plane(
    lo=(-0.006, 0.0, 0.04),
    hi=(0.008, 0.0, 0.05),
    normal=probe_normal,
    nx=8,
    ny=8,
)
tracked_cube_links = tuple(int(entity.base_link_idx) for entity in cube_entities.values())
tactile_options = dict(
    probe_local_pos=probe_local_pos,
    probe_local_normal=probe_normal,
    probe_radius=0.002,
    track_link_idx=tracked_cube_links,
    n_sample_points=1000,
    lambda_d=5000.0,
    lambda_s=4000.0,
    dilate_scale=1.0,
    shear_scale=1.0,
    normal_exponent=1.0,
    compressibility=0.8,
    draw_debug=False,
)
left_tactile = scene.add_sensor(
    gs.sensors.ElastomerTaxel(
        entity_idx=franka.idx,
        link_idx_local=franka.get_link("left_finger").idx_local,
        **tactile_options,
    )
)
right_tactile = scene.add_sensor(
    gs.sensors.ElastomerTaxel(
        entity_idx=franka.idx,
        link_idx_local=franka.get_link("right_finger").idx_local,
        **tactile_options,
    )
)

scene.build()
for _ in range(20):
    scene.step()

# Preserve the settled, upright Franka pose before later teaching cells move it.
UPRIGHT_QPOS = tensor_to_array(franka.get_qpos()).reshape(-1).astype(np.float64)
UPRIGHT_QPOS[-2:] = 0.04

print("Scene built with Genesis 1.3.1 APIs")
print("Semantic objects:", ", ".join(cube_entities))
print("Tactile pads: left 8×8 + right 8×8")
print("Saved upright reset pose:", UPRIGHT_QPOS.round(3).tolist())

## 2. Capture and inspect camera buffers

A Genesis camera can return several aligned modalities in one render call:

- **RGB** provides appearance and color.
- **Depth** stores the distance from the camera to visible surfaces.
- **Segmentation** assigns an integer ID to each rendered entity.
- **Normals** describe surface orientation at each pixel.

Before sending these arrays to PyTorch, we normalize possible batch and singleton-channel dimensions. Printing shape, dtype, and range is an important debugging step: it prevents a depth map or segmentation map from being interpreted as an ordinary RGB image.

In [ ]:
def to_numpy(value):
    """Convert Genesis/PyTorch output to a NumPy array."""
    if isinstance(value, torch.Tensor):
        return tensor_to_array(value)
    return np.asarray(value)


def squeeze_render_buffer(value):
    """Remove optional environment and singleton-channel dimensions."""
    arr = to_numpy(value)
    if arr.ndim == 4 and arr.shape[0] == 1:
        arr = arr[0]
    if arr.ndim == 3 and arr.shape[-1] == 1:
        arr = arr[..., 0]
    return arr


def as_rgb_uint8(value):
    """Normalize an RGB render buffer to H×W×3 uint8."""
    arr = squeeze_render_buffer(value)[..., :3]
    if arr.dtype != np.uint8:
        arr = arr.astype(np.float32)
        if arr.size and float(np.nanmax(arr)) <= 1.5:
            arr = arr * 255.0
        arr = np.clip(arr, 0, 255).astype(np.uint8)
    return arr


rgb_raw, depth_raw, seg_raw, normal_raw = cam.render(
    rgb=True,
    depth=True,
    segmentation=True,
    normal=True,
    colorize_seg=False,
)

rgb_u8 = as_rgb_uint8(rgb_raw)
depth_hw = squeeze_render_buffer(depth_raw).astype(np.float32)
seg_ids = squeeze_render_buffer(seg_raw).astype(np.int32)
normal_hwc = squeeze_render_buffer(normal_raw).astype(np.float32)

for label, arr in {
    "RGB": rgb_u8,
    "depth": depth_hw,
    "segmentation": seg_ids,
    "normal": normal_hwc,
}.items():
    print(f"{label:13s}: shape={arr.shape}, dtype={arr.dtype}, min={arr.min():.4g}, max={arr.max():.4g}")

plt.figure(figsize=(8, 6))
plt.imshow(rgb_u8)
plt.title("Genesis perception scene")
plt.axis("off")
plt.show()

## 3. Move camera data to ROCm

Each perception function follows the same boundary:

1. receive a NumPy camera buffer;
2. make its host memory contiguous;
3. create a PyTorch tensor and move it to `DEVICE`;
4. perform the expensive operation on the GPU;
5. copy only the final display result back to the CPU.

The source arrays stay alive until GPU work completes. We use `torch.from_numpy(...).to(DEVICE)` instead of DLPack so ownership and synchronization remain explicit and easy to inspect.

In [ ]:
_HOST_KEEPALIVE = collections.deque(maxlen=16)


def to_device(array, dtype=None):
    host = np.ascontiguousarray(array if dtype is None else np.asarray(array, dtype=dtype))
    _HOST_KEEPALIVE.append(host)
    return torch.from_numpy(host).to(DEVICE)


def depth_band(depth_np):
    depth = to_device(depth_np, np.float32)
    valid = torch.isfinite(depth) & (depth > 0)
    if not bool(valid.any()):
        return torch.zeros_like(depth, dtype=torch.uint8).cpu().numpy()
    values = depth[valid]
    near = torch.quantile(values, 0.02)
    far = torch.quantile(values, 0.98)
    scaled = ((depth - near) / (far - near + 1e-6)).clamp(0, 1)
    scaled = torch.where(valid, scaled, torch.zeros_like(scaled))
    return (scaled * 255).byte().cpu().numpy()


def normal_map(normal_np, segmentation_np=None):
    normal = to_device(normal_np, np.float32)
    if float(normal.max()) > 1.5:
        mapped = normal / 255.0  # Genesis rasterizer already encodes [-1, 1] as uint8 RGB.
    elif float(normal.min()) < -0.05:
        mapped = (normal + 1.0) / 2.0
    else:
        mapped = normal
    image = (mapped.clamp(0, 1) * 255).byte().cpu().numpy()
    if segmentation_np is not None:
        image[np.asarray(segmentation_np) == 0] = 0
    return image


def segmentation_map(segmentation_np):
    ids = to_device(segmentation_np, np.int32)
    r = ((ids * 37 + 11) % 256).byte()
    g = ((ids * 79 + 43) % 256).byte()
    b = ((ids * 131 + 97) % 256).byte()
    colored = torch.stack([r, g, b], dim=-1)
    colored[ids == 0] = 0
    return colored.cpu().numpy()


print(f"Kernel device: {DEVICE}")

### 3.1 Depth, normals, and segmentation

These first kernels transform geometry-oriented camera data:

- `depth_band()` maps the useful depth range to an 8-bit image. Percentiles reduce the effect of invalid pixels and distant outliers.
- `normal_map()` converts surface vectors into displayable RGB values.
- `segmentation_map()` hashes integer entity IDs into stable colors while keeping background ID `0` black.

They are visualization kernels: they help us inspect what the simulator sees before using the data to make a decision.

### 3.2 Locate a requested color

`color_lock()` turns RGB similarity into a small detection result. It computes a GPU mask, counts matching pixels, and returns a bounding box plus centroid only when enough evidence is present.

Returning a structured no-match result instead of raising an exception is important: GS06 can stop safely when an object is not visible.

In [ ]:
def color_lock(rgb_np, target_rgb, tolerance=0.30, min_pixels=8):
    """Find pixels close to target_rgb and summarize their image location."""
    rgb = to_device(np.asarray(rgb_np)[..., :3], np.float32) / 255.0
    target = torch.tensor(target_rgb, dtype=torch.float32, device=DEVICE)
    mask = torch.linalg.norm(rgb - target, dim=-1) < tolerance
    count = int(mask.sum().item())

    if count < min_pixels:
        return {
            "visible": False,
            "pixel_count": count,
            "bbox": None,
            "centroid_px": None,
            "mask": mask.cpu().numpy(),
        }

    ys, xs = torch.where(mask)
    return {
        "visible": True,
        "pixel_count": count,
        "bbox": [int(xs.min()), int(ys.min()), int(xs.max()), int(ys.max())],
        "centroid_px": [float(xs.float().mean()), float(ys.float().mean())],
        "mask": mask.cpu().numpy(),
    }

### 3.3 Build a classical image-processing chain

The remaining kernels demonstrate a common vision pipeline:

`RGB → grayscale → Gaussian blur → Sobel edges`

Grayscale reduces three color channels to one intensity value. Gaussian blur suppresses small variations, and Sobel filters measure horizontal and vertical intensity changes. Running the convolution operations through PyTorch keeps the expensive work on the ROCm device.

In [ ]:
def rgb_to_gray(rgb_np):
    rgb = to_device(np.asarray(rgb_np)[..., :3], np.float32)
    weights = torch.tensor([0.299, 0.587, 0.114], device=DEVICE)
    return (rgb * weights).sum(-1).clamp(0, 255).byte().cpu().numpy()


def gaussian_blur(gray_np, kernel_size=5, sigma=1.4):
    gray = to_device(gray_np, np.float32)[None, None]
    axis = torch.arange(kernel_size, device=DEVICE, dtype=torch.float32)
    axis = axis - (kernel_size - 1) / 2
    kernel = torch.exp(-(axis * axis) / (2 * sigma * sigma))
    kernel = kernel / kernel.sum()
    pad = kernel_size // 2
    out = F.conv2d(gray, kernel.view(1, 1, 1, -1), padding=(0, pad))
    out = F.conv2d(out, kernel.view(1, 1, -1, 1), padding=(pad, 0))
    return out[0, 0].clamp(0, 255).byte().cpu().numpy()


def sobel_edges(gray_np):
    gray = to_device(gray_np, np.float32)[None, None]
    gx = torch.tensor(
        [[-1, 0, 1], [-2, 0, 2], [-1, 0, 1]],
        dtype=torch.float32,
        device=DEVICE,
    ).view(1, 1, 3, 3)
    gy = gx.transpose(-1, -2)
    edge_x = F.conv2d(gray, gx, padding=1)
    edge_y = F.conv2d(gray, gy, padding=1)
    magnitude = torch.sqrt(edge_x.square() + edge_y.square())
    magnitude = magnitude / (magnitude.max() + 1e-6) * 255
    return magnitude[0, 0].byte().cpu().numpy()

### 3.4 Reduce two 8×8 tactile pads on ROCm

Each `ElastomerTaxel` returns an 8×8×3 marker-displacement field in meters. The final dimension describes local 3D displacement at one taxel.

`tactile_reduce()` flattens both fingertip grids and computes:

- the number of taxels above a contact threshold;
- total taxel count;
- a heuristic grip-force estimate;
- peak displacement in millimeters;
- a `secure` gate that becomes true only when enough taxels report contact.

The reduction runs on the same ROCm device as the vision kernels. The force scale is a teaching heuristic, while `secure` is the signal GS06 uses to decide whether lifting is allowed.

In [ ]:
def read_tactile_displacement():
    """Return left/right 8×8×3 ground-truth marker displacement in meters."""
    left = to_numpy(left_tactile.read_ground_truth()).astype(np.float32)
    right = to_numpy(right_tactile.read_ground_truth()).astype(np.float32)
    return left, right


def tactile_reduce(
    left_disp,
    right_disp,
    contact_thresh_m=CONTACT_THRESH_M,
    secure_taxels=CONTACT_SECURE_TAXELS,
    grip_stiffness_N_per_m=GRIP_STIFFNESS_N_PER_M,
):
    left = to_device(left_disp, np.float32).reshape(-1, 3)
    right = to_device(right_disp, np.float32).reshape(-1, 3)
    left_magnitude = torch.linalg.norm(left, dim=-1)
    right_magnitude = torch.linalg.norm(right, dim=-1)

    n_contact = int(
        (left_magnitude > contact_thresh_m).sum().item()
        + (right_magnitude > contact_thresh_m).sum().item()
    )
    peak_m = torch.maximum(left_magnitude.max(), right_magnitude.max())
    grip_force_N = (left_magnitude.sum() + right_magnitude.sum()) * grip_stiffness_N_per_m

    return {
        "n_contact": n_contact,
        "n_taxels": int(left_magnitude.numel() + right_magnitude.numel()),
        "grip_force_N": float(grip_force_N.item()),
        "peak_mm": float(peak_m.item() * 1000.0),
        "secure": bool(n_contact >= secure_taxels),
    }


scene.step()
air_tactile = tactile_reduce(*read_tactile_displacement())
print("Open-air tactile receipt:", air_tactile)
assert air_tactile["n_taxels"] == 128
assert air_tactile["secure"] is False

## 4. Run the perception pipeline

Rendered colors differ from ideal RGB values because lighting and materials affect pixel intensity. We therefore use entity-level segmentation once to estimate the rendered mean color of each cube. The runtime detector then relies only on RGB similarity.

The deliberately absent yellow target checks the failure path: it should return `visible=False` without interrupting the notebook.

In [ ]:
# Calibrate the rendered target colors from entity-level segmentation.
# Genesis reserves segmentation ID 0 for the background; entity IDs are idx + 1.
target_colors = {}
for name, entity in cube_entities.items():
    segmentation_id = int(entity.idx + 1)
    entity_mask = seg_ids == segmentation_id
    if entity_mask.any():
        target_colors[name] = (rgb_u8[entity_mask].mean(axis=0) / 255.0).tolist()
        source = "rendered pixels selected by segmentation"
    else:
        target_colors[name] = list(cube_specs[name]["color"][:3])
        source = "nominal surface color fallback"
    print(f"{name:5s}: seg_id={segmentation_id}, pixels={int(entity_mask.sum())}, source={source}")

processed = {
    "depth": depth_band(depth_hw),
    "normal": normal_map(normal_hwc, seg_ids),
    "segmentation": segmentation_map(seg_ids),
}
processed["gray"] = rgb_to_gray(rgb_u8)
processed["blur"] = gaussian_blur(processed["gray"])
processed["sobel"] = sobel_edges(processed["blur"])

lock_results = {
    name: color_lock(rgb_u8, target_rgb)
    for name, target_rgb in target_colors.items()
}
lock_results["yellow"] = color_lock(rgb_u8, [1.0, 1.0, 0.0])

for name, result in lock_results.items():
    printable = {key: value for key, value in result.items() if key != "mask"}
    print(f"{name:6s} -> {printable}")

## 5. Compare the perception outputs

The eight-panel dashboard places raw sensing and processed results side by side. This makes it easier to answer three questions:

1. Is the object visible in the original RGB frame?
2. Do geometry buffers and edge filters contain the expected structure?
3. Does the selected color mask cover only the requested cube?

The bounding box and centroid are overlaid on the original RGB frame so the detection can be checked visually.

In [ ]:
REQUESTED_TARGET = "green"
selected_lock = lock_results[REQUESTED_TARGET]
lock_panel = np.zeros_like(rgb_u8)
lock_panel[selected_lock["mask"]] = rgb_u8[selected_lock["mask"]]

fig, axes = plt.subplots(2, 4, figsize=(16, 8))
panels = [
    (rgb_u8, "RGB camera", None),
    (processed["depth"], "Depth band", "inferno"),
    (processed["normal"], "Normal map", None),
    (processed["segmentation"], "Segmentation", None),
    (lock_panel, f"Color lock: {REQUESTED_TARGET}", None),
    (processed["gray"], "Grayscale", "gray"),
    (processed["blur"], "Gaussian blur", "gray"),
    (processed["sobel"], "Sobel edges", "magma"),
]

for axis, (image, title, cmap) in zip(axes.ravel(), panels):
    axis.imshow(image, cmap=cmap)
    axis.set_title(title)
    axis.axis("off")

if selected_lock["visible"]:
    x0, y0, x1, y1 = selected_lock["bbox"]
    cx, cy = selected_lock["centroid_px"]
    axes[0, 0].add_patch(
        mpatches.Rectangle(
            (x0, y0), x1 - x0 + 1, y1 - y0 + 1,
            fill=False, edgecolor="red", linewidth=2,
        )
    )
    axes[0, 0].plot(cx, cy, "+", color="yellow", markersize=14, markeredgewidth=2)

artifact_path = "Artifacts/gs05_perception.png"
plt.tight_layout()
plt.savefig(artifact_path, dpi=150)
plt.show()
print("Saved:", artifact_path)

## 6. Create a target record for the action layer

A detector result contains image-space evidence, but the robot action layer needs a 3D target. For this teaching bridge:

- visibility, pixel count, bounding box, and centroid come from the RGB detector;
- world position comes from `entity.get_pos(relative=False)`;
- `position_source` explicitly labels that position as simulator state.

This provenance field prevents downstream code from presenting perfect simulator coordinates as if they had been reconstructed from a monocular camera.

In [ ]:
def entity_world_position(entity):
    position = to_numpy(entity.get_pos(relative=False)).reshape(-1)[:3]
    return position.astype(float).tolist()


def build_target_record(name, lock):
    visible = bool(lock.get("visible", False)) and name in cube_entities
    return {
        "name": name,
        "visible": visible,
        "pixel_count": int(lock.get("pixel_count", 0)),
        "bbox": lock.get("bbox"),
        "centroid_px": lock.get("centroid_px"),
        "world_position": entity_world_position(cube_entities[name]) if visible else None,
        "position_source": "genesis_entity_state" if visible else None,
    }


target_record = build_target_record(REQUESTED_TARGET, selected_lock)
missing_record = build_target_record("yellow", lock_results["yellow"])

print("Selected target record:")
print(json.dumps(target_record, indent=2))
print("\nMissing target record:")
print(json.dumps(missing_record, indent=2))

assert target_record["visible"], "the selected cube must be visible before handoff"
assert target_record["world_position"] is not None
assert target_record["position_source"] == "genesis_entity_state"
assert missing_record["visible"] is False
assert missing_record["world_position"] is None

## 7. Demonstrate a tactile-secure grasp

The open-air receipt above should contain no secure contact. We now move the gripper to the centered green cube and close it while reading both pads after every simulation step. The centered target uses the same stable workspace demonstrated in GS03.

The loop stops when `secure=True` or when the timeout is reached. The two heatmaps show displacement magnitude across the left and right 8×8 pads. This is the tactile equivalent of visually inspecting a segmentation mask.

In [ ]:
motors_dof = np.arange(7)
fingers_dof = np.arange(7, 9)
end_effector = franka.get_link("hand")

franka.set_dofs_kp(
    np.array([4500, 4500, 3500, 3500, 2000, 2000, 2000, 100, 100])
)
franka.set_dofs_kv(
    np.array([450, 450, 350, 350, 200, 200, 200, 10, 10])
)
franka.set_dofs_force_range(
    np.array([-87, -87, -87, -87, -12, -12, -12, -100, -100]),
    np.array([87, 87, 87, 87, 12, 12, 12, 100, 100]),
)

# Match the official Genesis tactile_franka.py workspace exactly.
cube_entities["green"].set_pos((0.5, 0.1, CUBE_Z))
for _ in range(10):
    scene.step()

grasp_center = np.asarray(entity_world_position(cube_entities["green"]))
pre_grasp = np.array([grasp_center[0], grasp_center[1], 0.18])
grasp = np.array([grasp_center[0], grasp_center[1], 0.125])
lift = np.array([grasp_center[0], grasp_center[1], 0.30])

pre_qpos = to_numpy(
    franka.inverse_kinematics(
        link=end_effector,
        pos=pre_grasp,
        quat=np.array([0.0, 1.0, 0.0, 0.0]),
        dofs_idx_local=motors_dof,
    )
).reshape(-1)

grasp_qpos = to_numpy(
    franka.inverse_kinematics(
        link=end_effector,
        pos=grasp,
        quat=np.array([0.0, 1.0, 0.0, 0.0]),
        init_qpos=pre_qpos,
        dofs_idx_local=motors_dof,
    )
).reshape(-1)
lift_qpos = to_numpy(
    franka.inverse_kinematics(
        link=end_effector,
        pos=lift,
        quat=np.array([0.0, 1.0, 0.0, 0.0]),
        init_qpos=grasp_qpos,
        dofs_idx_local=motors_dof,
    )
).reshape(-1)
retreat_qpos = to_numpy(
    franka.inverse_kinematics(
        link=end_effector,
        pos=pre_grasp,
        quat=np.array([0.0, 1.0, 0.0, 0.0]),
        init_qpos=grasp_qpos,
        dofs_idx_local=motors_dof,
    )
).reshape(-1)


def require_nearby_joint_goal(current, goal, label, max_delta=0.5):
    delta = np.abs(np.asarray(goal)[motors_dof] - np.asarray(current)[motors_dof])
    if float(delta.max()) > max_delta:
        raise RuntimeError(
            f"{label} crosses IK branches: max joint delta={float(delta.max()):.3f} rad"
        )


require_nearby_joint_goal(pre_qpos, grasp_qpos, "pre→grasp")
require_nearby_joint_goal(grasp_qpos, lift_qpos, "grasp→lift")
require_nearby_joint_goal(grasp_qpos, retreat_qpos, "grasp→retreat")

preview_path = "Videos/video_05.mp4"
hud_path = "Videos/video_05_hud.mp4"
contact_tactile = None
best_tactile = {"n_contact": -1}
contact_snapshot = None

# Establish tactile-secure contact before starting the camera and HUD recordings.
franka.set_qpos(pre_qpos[motors_dof], motors_dof)
franka.control_dofs_position(np.array([0.04, 0.04]), fingers_dof)
for _ in range(30):
    scene.step()

franka.control_dofs_position(grasp_qpos[motors_dof], motors_dof)
for _ in range(100):
    scene.step()

for step in range(150):
    franka.control_dofs_position(grasp_qpos[motors_dof], motors_dof)
    franka.control_dofs_position(np.array([-0.03, -0.03]), fingers_dof)
    scene.step()
    displacement = read_tactile_displacement()
    contact_tactile = tactile_reduce(*displacement)
    if contact_tactile["n_contact"] > best_tactile["n_contact"]:
        best_tactile = contact_tactile.copy()
        contact_snapshot = displacement
    if contact_tactile["secure"]:
        break

assert contact_tactile["secure"], (
    "expected the fingertip taxels to confirm a secure grasp; "
    f"best receipt was {best_tactile}"
)

gpu_monitor = GPUMonitor().start()
hud_writer = FFmpegHUDWriter(hud_path, fps=25).open()
hud_frame_index = 0
cam.start_recording(save_to_filename=preview_path, fps=25)


def record_frame(status):
    global hud_frame_index
    hud_frame_index += 1
    if hud_frame_index % 4 != 0:
        cam.render(rgb=True, depth=False, segmentation=False, normal=False)
        return

    rgb_frame, depth_frame, seg_frame, normal_frame = cam.render(
        rgb=True,
        depth=True,
        segmentation=True,
        normal=True,
        colorize_seg=False,
    )
    rgb_now = as_rgb_uint8(rgb_frame)
    depth_now = squeeze_render_buffer(depth_frame).astype(np.float32)
    seg_now = squeeze_render_buffer(seg_frame).astype(np.int32)
    normal_now = squeeze_render_buffer(normal_frame).astype(np.float32)
    thumbnails_now = {
        "depth": depth_band(depth_now),
        "normal": normal_map(normal_now, seg_now),
        "segmentation": segmentation_map(seg_now),
    }
    thumbnails_now["gray"] = rgb_to_gray(rgb_now)
    thumbnails_now["blur"] = gaussian_blur(thumbnails_now["gray"])
    thumbnails_now["sobel"] = sobel_edges(thumbnails_now["blur"])

    left_now, right_now = read_tactile_displacement()
    tactile_now = tactile_reduce(left_now, right_now)
    hud_writer.write(
        compose_hud_frame(
            rgb_now,
            title="GS05 · Multimodal Perception",
            status=status,
            user_input="Inspect the centered cube and verify tactile contact",
            plan={"lesson": "vision + tactile", "target": "green"},
            stages=[{"stage": "perception", "success": True}],
            thumbnails=thumbnails_now,
            tactile=tactile_now,
            left_tactile=left_now,
            right_tactile=right_now,
            gpu=gpu_monitor.snapshot(),
        )
    )


try:
    # Hold the secure grasp for one second, then lift, lower, release, and retreat.
    for _ in range(100):
        scene.step()
        record_frame("Tactile secure · holding before lift")

    # Lift, hold for inspection, lower, release, and retreat.
    franka.control_dofs_position(lift_qpos[motors_dof], motors_dof)
    for _ in range(150):
        scene.step()
        record_frame("Lifting only after K4 secure contact")
    for _ in range(100):
        scene.step()
        record_frame("Holding the cube · monitor both tactile pads")

    franka.control_dofs_position(grasp_qpos[motors_dof], motors_dof)
    for _ in range(150):
        scene.step()
        record_frame("Lowering the cube while maintaining contact")

    franka.control_dofs_position(np.array([0.04, 0.04]), fingers_dof)
    for _ in range(100):
        scene.step()
        record_frame("Opening gripper · contact should return to zero")

    franka.control_dofs_position(retreat_qpos[motors_dof], motors_dof)
    for _ in range(120):
        scene.step()
        record_frame("Retreating on the same IK branch")
finally:
    cam.stop_recording()
    hud_writer.close()
    gpu_monitor.stop()

print(f"Tactile gate after {step + 1} close steps:", contact_tactile)
print("Saved complete grasp sequence:", preview_path)
print(f"Saved {hud_writer.frames_written} HUD frames:", hud_path)

left_disp, right_disp = contact_snapshot
fig, axes = plt.subplots(1, 2, figsize=(8, 3))
for axis, displacement, title in zip(
    axes,
    (left_disp, right_disp),
    ("Left fingertip |displacement|", "Right fingertip |displacement|"),
):
    heatmap = np.linalg.norm(displacement, axis=-1) * 1000.0
    image = axis.imshow(heatmap, cmap="magma", vmin=0)
    axis.set_title(title)
    axis.set_xlabel("taxel x")
    axis.set_ylabel("taxel y")
    fig.colorbar(image, ax=axis, label="mm")
plt.tight_layout()
plt.show()

## 8. Review the complete grasp sequence

The recording now covers 7.2 simulated seconds at 25 FPS:

`tactile-secure hold → lift → hold → lower → release → retreat`

This complete sequence makes the relationship between tactile confirmation and robot motion visible. The gripper does not lift until the secure-contact assertion passes.

Two videos are produced: a raw Genesis camera view and a 1440×816 HUD that adds GPU telemetry, six ROCm vision outputs, live left/right tactile heatmaps, and the current K4 secure state.

In [ ]:
assert os.path.exists(preview_path), "Run the tactile-grasp demonstration before displaying the video."
assert os.path.exists(hud_path), "The HUD recording was not created."
print("Saved complete tactile-grasp video:", preview_path)
print("Saved multimodal HUD video:", hud_path)

In [ ]:
from IPython.display import Video, display

print("Raw Genesis camera recording")
display(Video("Videos/video_05.mp4", embed=True, width=720))

print("Multimodal HUD recording")
display(Video("Videos/video_05_hud.mp4", embed=True, width=960))

## 9. Live simulation interface

The widget below controls the existing Genesis scene directly; it is not a replay of the MP4. Each button advances the simulation, reads the current camera and both 8×8 tactile pads, and refreshes the same multimodal HUD used by `video_05_hud.mp4`.

Recommended sequence:

1. Select a target and press **Reset**.
2. Press **Approach**.
3. Press **Close to Secure** and watch the K4 banner plus tactile heatmaps.
4. Press **Lift** only after secure contact.
5. Press **Lower**, then **Release**.
6. Press **Export HUD MP4** to save the captured interaction to `Videos/video_05_live_hud.mp4`.
7. Press **Shutdown HUD** when finished to stop the GPU telemetry thread.

The contact threshold and required secure-taxel count are live controls. If a grasp does not become secure, adjust them deliberately and compare the heatmaps rather than bypassing the gate.

In [ ]:
from IPython.display import display

live_monitor = GPUMonitor()  # starts on the first button action
live_state = {
    "target": None,
    "layout": None,
    "layout_seed": None,
    "layout_positions": None,
    "pre_qpos": None,
    "grasp_qpos": None,
    "lift_qpos": None,
    "secure": False,
    "tactile": air_tactile,
}

RANDOM_LAYOUT_X = (0.46, 0.62)
RANDOM_LAYOUT_Y = (-0.20, 0.20)
RANDOM_LAYOUT_MIN_DISTANCE = 0.09


def live_cube_layout_positions(controller):
    """Return deterministic default or seeded-random cube positions."""
    if controller.scene_layout.value == "default":
        return {
            name: np.asarray(spec["pos"], dtype=np.float64).copy()
            for name, spec in cube_specs.items()
        }

    rng = np.random.default_rng(int(controller.layout_seed.value))
    positions = {}
    for name in cube_entities:
        for _ in range(200):
            candidate = np.array(
                [
                    rng.uniform(*RANDOM_LAYOUT_X),
                    rng.uniform(*RANDOM_LAYOUT_Y),
                    CUBE_Z,
                ],
                dtype=np.float64,
            )
            if all(
                np.linalg.norm(candidate[:2] - other[:2]) >= RANDOM_LAYOUT_MIN_DISTANCE
                for other in positions.values()
            ):
                positions[name] = candidate
                break
        else:
            raise RuntimeError("Could not sample a collision-free seeded cube layout")
    return positions


def live_prepare_target(controller):
    """Apply the selected layout and compute Franka tactile-demo poses."""
    selected = controller.target.value
    layout_positions = live_cube_layout_positions(controller)
    for name, entity in cube_entities.items():
        entity.set_pos(layout_positions[name])
    for _ in range(10):
        scene.step()

    center = np.asarray(entity_world_position(cube_entities[selected]))
    positions = {
        "pre": np.array([center[0], center[1], 0.18]),
        "grasp": np.array([center[0], center[1], 0.125]),
        "lift": np.array([center[0], center[1], 0.30]),
    }
    pre_qpos = to_numpy(
        franka.inverse_kinematics(
            link=end_effector,
            pos=positions["pre"],
            quat=np.array([0.0, 1.0, 0.0, 0.0]),
            init_qpos=UPRIGHT_QPOS,
            dofs_idx_local=motors_dof,
        )
    ).reshape(-1)
    grasp_qpos = to_numpy(
        franka.inverse_kinematics(
            link=end_effector,
            pos=positions["grasp"],
            quat=np.array([0.0, 1.0, 0.0, 0.0]),
            init_qpos=pre_qpos,
            dofs_idx_local=motors_dof,
        )
    ).reshape(-1)
    lift_qpos = to_numpy(
        franka.inverse_kinematics(
            link=end_effector,
            pos=positions["lift"],
            quat=np.array([0.0, 1.0, 0.0, 0.0]),
            init_qpos=grasp_qpos,
            dofs_idx_local=motors_dof,
        )
    ).reshape(-1)
    retreat_qpos = to_numpy(
        franka.inverse_kinematics(
            link=end_effector,
            pos=positions["pre"],
            quat=np.array([0.0, 1.0, 0.0, 0.0]),
            init_qpos=grasp_qpos,
            dofs_idx_local=motors_dof,
        )
    ).reshape(-1)
    require_nearby_joint_goal(pre_qpos, grasp_qpos, "live pre→grasp")
    require_nearby_joint_goal(grasp_qpos, lift_qpos, "live grasp→lift")
    require_nearby_joint_goal(grasp_qpos, retreat_qpos, "live grasp→retreat")

    live_state.update(
        {
            "target": selected,
            "layout": controller.scene_layout.value,
            "layout_seed": int(controller.layout_seed.value),
            "layout_positions": {
                name: position.copy() for name, position in layout_positions.items()
            },
            "pre_qpos": pre_qpos,
            "grasp_qpos": grasp_qpos,
            "lift_qpos": lift_qpos,
            "retreat_qpos": retreat_qpos,
            "secure": False,
        }
    )


def live_update(controller, status, *, capture=True):
    live_monitor.start()
    rgb_frame, depth_frame, seg_frame, normal_frame = cam.render(
        rgb=True,
        depth=True,
        segmentation=True,
        normal=True,
        colorize_seg=False,
    )
    rgb_now = as_rgb_uint8(rgb_frame)
    depth_now = squeeze_render_buffer(depth_frame).astype(np.float32)
    seg_now = squeeze_render_buffer(seg_frame).astype(np.int32)
    normal_now = squeeze_render_buffer(normal_frame).astype(np.float32)
    thumbnails_now = {
        "depth": depth_band(depth_now),
        "normal": normal_map(normal_now, seg_now),
        "segmentation": segmentation_map(seg_now),
    }
    thumbnails_now["gray"] = rgb_to_gray(rgb_now)
    thumbnails_now["blur"] = gaussian_blur(thumbnails_now["gray"])
    thumbnails_now["sobel"] = sobel_edges(thumbnails_now["blur"])

    left_now, right_now = read_tactile_displacement()
    tactile_now = tactile_reduce(
        left_now,
        right_now,
        contact_thresh_m=controller.contact_threshold.value,
        secure_taxels=controller.secure_taxels.value,
    )
    live_state["tactile"] = tactile_now
    frame = compose_hud_frame(
        rgb_now,
        title="GS05 · Live Multimodal Perception",
        status=status,
        user_input=f"Interactive target: {controller.target.value}",
        plan={"action": "tactile grasp lab", "target": controller.target.value},
        stages=[{"stage": "live simulation", "success": tactile_now["secure"]}],
        thumbnails=thumbnails_now,
        tactile=tactile_now,
        left_tactile=left_now,
        right_tactile=right_now,
        gpu=live_monitor.snapshot(),
    )
    controller.update(frame, status, capture=capture)
    return tactile_now


def live_steps(controller, count, status, *, every=4):
    for index in range(count):
        scene.step()
        if index % every == 0:
            live_update(controller, status)
    return live_update(controller, status)

In [ ]:
def live_follow_planned_path(controller, qpos_goal, status, *, waypoints):
    """Plan and execute a smooth, joint-limit-checked path for the live HUD."""
    goal = np.asarray(qpos_goal, dtype=np.float64).copy()
    goal[-2:] = 0.04
    path, path_valid = franka.plan_path(
        qpos_goal=goal,
        num_waypoints=waypoints,
        return_valid_mask=True,
    )
    if not bool(to_numpy(path_valid).reshape(-1)[0]):
        raise RuntimeError(f"Motion planning failed while {status.lower()}")
    for index, waypoint in enumerate(path):
        franka.control_dofs_position(waypoint)
        scene.step()
        if index % 4 == 0:
            live_update(controller, status)
    return live_update(controller, status)


def live_reset(controller):
    franka.set_qpos(UPRIGHT_QPOS)
    franka.control_dofs_position(UPRIGHT_QPOS[motors_dof], motors_dof)
    franka.control_dofs_position(np.array([0.04, 0.04]), fingers_dof)
    live_prepare_target(controller)
    layout_status = (
        "Default layout"
        if controller.scene_layout.value == "default"
        else f"Random layout · seed {controller.layout_seed.value}"
    )
    live_steps(controller, 30, f"Reset · upright home · {layout_status}")


def live_approach(controller):
    layout_changed = (
        live_state["layout"] != controller.scene_layout.value
        or (
            controller.scene_layout.value == "random"
            and live_state["layout_seed"] != int(controller.layout_seed.value)
        )
    )
    if live_state["target"] != controller.target.value or layout_changed:
        live_prepare_target(controller)
    live_follow_planned_path(
        controller,
        live_state["pre_qpos"],
        f"Moving to pre-grasp for {controller.target.value}",
        waypoints=160,
    )
    live_follow_planned_path(
        controller,
        live_state["grasp_qpos"],
        f"Approaching {controller.target.value}",
        waypoints=100,
    )
    live_state["secure"] = False


def live_close(controller):
    if live_state["grasp_qpos"] is None:
        live_prepare_target(controller)
    receipt = None
    for step in range(150):
        franka.control_dofs_position(live_state["grasp_qpos"][motors_dof], motors_dof)
        franka.control_dofs_position(np.array([-0.03, -0.03]), fingers_dof)
        scene.step()
        if step % 4 == 0:
            receipt = live_update(
                controller,
                f"Closing · contact {live_state['tactile']['n_contact']}/{128}",
            )
        if receipt is not None and receipt["secure"]:
            break
    receipt = live_update(controller, "K4 secure" if receipt and receipt["secure"] else "K4 timeout")
    live_state["secure"] = bool(receipt["secure"])
    if not live_state["secure"]:
        raise RuntimeError(f"Tactile gate did not become secure: {receipt}")


def live_lift(controller):
    if not live_state["secure"]:
        raise RuntimeError("Lift blocked: press Close to Secure first")
    franka.control_dofs_position(live_state["lift_qpos"][motors_dof], motors_dof)
    live_steps(controller, 150, "Lifting after tactile secure")


def live_lower(controller):
    franka.control_dofs_position(live_state["grasp_qpos"][motors_dof], motors_dof)
    live_steps(controller, 150, "Lowering while monitoring tactile contact")


def live_release(controller):
    franka.control_dofs_position(np.array([0.04, 0.04]), fingers_dof)
    live_steps(controller, 100, "Released · tactile contact returns to zero")
    live_state["secure"] = False
    franka.control_dofs_position(live_state["retreat_qpos"][motors_dof], motors_dof)
    live_steps(controller, 120, "Retreating on the same IK branch")


def live_shutdown(controller):
    live_monitor.stop()
    controller.set_status("Live HUD monitor stopped. Any action button will restart it.")


live_hud = LiveHUDController(
    targets=tuple(cube_entities),
    contact_threshold=CONTACT_THRESH_M,
    secure_taxels=CONTACT_SECURE_TAXELS,
    export_path="Videos/video_05_live_hud.mp4",
    export_fps=25,
)
live_hud.bind("reset", live_reset)
live_hud.bind("approach", live_approach)
live_hud.bind("close", live_close)
live_hud.bind("lift", live_lift)
live_hud.bind("lower", live_lower)
live_hud.bind("release", live_release)
live_hud.bind("shutdown", live_shutdown)

live_update(
    live_hud,
    "Live preview · press Reset to initialize the selected target.",
    capture=False,
)
display(live_hud.widget)

## 10. Experiment further

1. **Detection sensitivity:** change the color-lock tolerance and record false positives and false negatives.
2. **Spatial response:** move one cube, render again, and compare its bounding box and centroid.
3. **Noise reduction:** add image noise and compare Sobel output before and after Gaussian blur.
4. **Tactile sensitivity:** change `CONTACT_THRESH_M` or `CONTACT_SECURE_TAXELS` and compare open-air and grasp receipts.
5. **3D extension:** use camera intrinsics, extrinsics, and depth to estimate a world position; compare it with `entity.get_pos(relative=False)`.

### Handoff to GS06

GS06 reuses the visual target record and the tactile receipt. It adds constrained language planning and allows lifting only after the same tactile reduction reports `secure=True`.

## Conclusions

You combined four Genesis camera modalities with two 8×8 fingertip tactile pads, processed both through editable ROCm/PyTorch reductions, and produced explicit visual and tactile contracts. The live widget lets you control each grasp phase, tune contact thresholds, inspect synchronized heatmaps, and export the interaction as `video_05_live_hud.mp4`. GS06 consumes the same contracts to build a language-guided agent that lifts only after contact is secure.

## Acknowledgements

This notebook adapts the ROCm perception-kernel teaching approach from `AI_LABS/vision_kernels_rocm/ROCm_Physical_AI_Agent_Workshop.ipynb` in the original [ssw-mktg/igpu-training-env](https://gitenterprise.xilinx.com/ssw-mktg/igpu-training-env) repository.

An earlier version of this material was presented as a demonstration and used for teaching at Advancing AI 2026. It was subsequently refined and adapted into the current notebook and Genesis 1.3.1 course environment.

We thank the original repository contributors for the Physical AI vision workshop material that provided the foundation for this Genesis 1.3.1 course adaptation.

---

Copyright (C) 2026 Advanced Micro Devices, Inc. All rights reserved. Portions of this file consist of AI-generated content.  
SPDX-License-Identifier: MIT